In [58]:
def _conj_root_dep(token) -> str:
    """Follow conj chain to find ultimate head's dep."""
    current = token
    while current.dep_ == "conj":
        current = current.head
    return current.dep_

def _is_inside_parentheses(sent, comma_token) -> bool:
    """Check if a comma is inside parentheses."""
    depth = 0
    for token in sent:
        if token.text in ("(", "[", "{"):
            depth += 1
        elif token.text in (")", "]", "}"):
            depth -= 1
        if token.i == comma_token.i and depth > 0:
            return True
    return False

_PARENTHETICAL = re.compile(
    r"\b(for example|i\.e|e\.g|for instance|namely)\s*$", re.I
)

def has_splittable_coordination(sent) -> bool:
    """Check if a sentence has a coordination structure that can be split."""
    conj_tokens = [t for t in sent if t.dep_ == "conj"]
    if not conj_tokens:
        return False
    for token in conj_tokens:
        if token.pos_ in ("ADJ", "ADV"):
            continue
        ultimate_dep = _conj_root_dep(token)
        if ultimate_dep in ("pobj", "dobj", "attr", "acomp", "xcomp"):
            continue
        return True
    return False

def split_by_coordination(text: str, nlp) -> list[str]:
    """
    Split a sentence on comma/conjunction boundaries using spaCy
    dependency relations to validate each split point.
    """
    # Never split sentences with colons
    if ":" in text:
        return [text]

    doc = nlp(text)
    sent = list(doc.sents)[0]

    # Step 1: Check if sentence has splittable coordination
    if not has_splittable_coordination(sent):
        return [text]

    # Step 2: Find valid comma split points
    comma_tokens = [t for t in sent if t.text == ","]
    if not comma_tokens:
        return [text]

    valid_commas = []
    step2_prev = 0  # Only used for parenthetical detection in Step 2

    for comma in comma_tokens:
        # Skip commas inside parentheses
        if _is_inside_parentheses(sent, comma):
            step2_prev = comma.i + 1
            continue

        # Skip commas preceded by preposition (e.g. "cooperate with,")
        if comma.i > 0 and sent[comma.i - 1].pos_ == "ADP":
            step2_prev = comma.i + 1
            continue

        # Skip commas after parenthetical expressions
        if _PARENTHETICAL.search(sent[step2_prev:comma.i].text):
            step2_prev = comma.i + 1
            continue

        # Find token after comma (skip conjunctions and auxiliaries)
        right_i = comma.i + 1
        while right_i < len(sent) and sent[right_i].pos_ in ("CCONJ", "AUX"):
            right_i += 1
        if right_i >= len(sent):
            continue
        right = sent[right_i]

        # Right token must be conj
        if right.dep_ != "conj":
            step2_prev = comma.i + 1
            continue

        # Right token cannot be adjective, adverb, or number
        if right.pos_ in ("ADJ", "ADV", "NUM"):
            step2_prev = comma.i + 1
            continue

        # conj chain root cannot share context
        ultimate_dep = _conj_root_dep(right)
        if ultimate_dep in ("pobj", "dobj", "attr", "acomp", "xcomp"):
            step2_prev = comma.i + 1
            continue

        # Right token must have its own syntactic structure
        has_own_structure = any(
            t.dep_ in (
                "dobj", "nsubj", "prep", "attr", "ccomp",
                "relcl", "advmod", "acl", "xcomp", "oprd"
            )
            for t in right.children
        )
        if not has_own_structure:
            step2_prev = comma.i + 1
            continue

        valid_commas.append(comma.i)
        step2_prev = comma.i + 1

    if not valid_commas:
        return [text]

    # Step 3: Split on valid commas only
    parts = []
    prev = 0
    for comma_i in valid_commas:
        part = sent[prev:comma_i].text.strip()
        part = re.sub(r"^(and|or|but)\s+", "", part, flags=re.I)
        if part:
            parts.append(part)
        prev = comma_i + 1

    last = sent[prev:].text.strip()
    last = re.sub(r"^(and|or|but)\s+", "", last, flags=re.I)
    if last:
        parts.append(last)

    if len(parts) < 2:
        return [text]

    return parts

In [ ]:
# 重新测试
total = 0
split_accepted = 0
split_examples = []

for raw in samples:
    cleaned = clean_description(raw)
    sentences = nltk_split(cleaned)
    
    for text in sentences:
        parts = split_by_coordination(text, nlp)
        total += 1
        if len(parts) > 1:
            split_accepted += 1
            split_examples.append((text, parts))

print(f"总句数: {total}")
print(f"成功切分: {split_accepted} ({split_accepted/total*100:.0f}%)")

import random
random.seed(42)
sample_examples = random.sample(split_examples, min(15, len(split_examples)))

print(f"\n=== 随机抽取{len(sample_examples)}个切分例子 ===")
for text, parts in sample_examples:
    print(f"\n原句: {text}")
    for p in parts:
        print(f"  → {p}")

总句数: 497
成功切分: 17 (3%)

=== 随机抽取15个切分例子 ===

原句: accurately and efficiently vital signs, obtaining specimens, and supporting the Physician with any other diagnostic tests.
  → accurately and efficiently vital signs, obtaining specimens
  → supporting the Physician with any other diagnostic tests.

原句: Work involves assigned duties that require prioritization of tasks, selection of appropriate equipment, selection of the appropriate procedure for assigned treatments, and recognition of differences in resident's medical condition to facilitate appropriate and accurate documentation.
  → Work involves assigned duties that require prioritization of tasks, selection of appropriate equipment, selection of the appropriate procedure for assigned treatments
  → recognition of differences in resident's medical condition to facilitate appropriate and accurate documentation.

原句: For example, recognizing need for basic life support, controlling bleeding and assisting with behavior crisis, etc.
  →

: 

In [57]:
text = "For example, recognizing need for basic life support, controlling bleeding."
doc = nlp(text)
sent = list(doc.sents)[0]

prev = 0
for comma in [t for t in sent if t.text == ","]:
    segment = sent[prev:comma.i].text
    match = _PARENTHETICAL.search(segment)
    print(f"逗号{comma.i}: segment={repr(segment)} parenthetical={bool(match)}")
    prev = comma.i + 1

逗号2: segment='For example' parenthetical=True
逗号9: segment='recognizing need for basic life support' parenthetical=False
